
[![](imagens/colab-badge.png){width="16%"}](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cap03/cap03_aluno.ipynb)
[![](imagens/github-badge.png){width="20%"}](https://github.com/fzampirolli/pdi-vc)

# Operações Espaciais: Filtragem, Convolução e Detecção de Bordas

Este capítulo introduz as operações espaciais fundamentais do Processamento Digital de Imagens (PDI), com foco em filtragem, convolução, suavização, realce e detecção de bordas.

As técnicas apresentadas são a base de inúmeras aplicações em:
- redução de ruído;
- realce de detalhes;
- segmentação;
- visão computacional;
- reconhecimento de padrões;
- inteligência artificial aplicada a imagens.

A abordagem segue uma linha prática e experimental, utilizando Python, NumPy e OpenCV.



## Objetivos

Ao final deste capítulo, você será capaz de:

- Compreender o conceito de vizinhança espacial em imagens digitais;
- Aplicar convolução bidimensional;
- Interpretar o funcionamento de kernels e filtros espaciais;
- Utilizar filtros de suavização e nitidez;
- Detectar bordas utilizando gradientes;
- Comparar diferentes estratégias de redução de ruído;
- Implementar operações básicas utilizando NumPy e OpenCV.



## Introdução às Operações Espaciais

Uma operação espacial modifica o valor de um pixel considerando os valores de seus vizinhos.

Essas operações normalmente utilizam uma pequena matriz chamada:

- máscara;
- kernel;
- filtro;
- janela de convolução.

A ideia central consiste em percorrer a imagem aplicando operações locais sobre regiões vizinhas.



## Convolução Bidimensional

A convolução bidimensional é uma das operações mais importantes do PDI.

Seja:

$$
f(x,y)
$$

uma imagem de entrada e:

$$
w(s,t)
$$

um kernel de convolução.

A saída é dada por:

$$
g(x,y)=\sum_{s=-a}^{a}\sum_{t=-b}^{b} w(s,t)f(x-s,y-t)
$$

Essa operação permite implementar:
- suavização;
- realce;
- detecção de bordas;
- filtragem passa-baixa;
- filtragem passa-alta.


In [ ]:

#| quarto-raw: true

import os
import importlib
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
import cv2

# Baixar morph.py se necessário
BASE_URL = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/"

if not os.path.exists("morph.py"):
    urllib.request.urlretrieve(BASE_URL + "notebooks/morph.py", "morph.py")

import morph as mm

plt.rcParams["figure.figsize"] = (10, 6)



## Carregando uma imagem de exemplo

Utilizaremos uma imagem clássica do PDI para os experimentos deste capítulo.


In [ ]:

#| quarto-raw: true

url_lena = "https://upload.wikimedia.org/wikipedia/en/7/7d/Lenna_%28test_image%29.png"

img_color = mm.read(url_lena)
img_gray = mm.gray(img_color)

print("Imagem:", img_gray.shape)


In [ ]:

#| label: fig-03-imagem-original
#| fig-cap: "Imagem original em tons de cinza."
#| echo: true
#| output: true

plt.imshow(img_gray, cmap='gray')
plt.axis('off')
plt.show()



## Kernels e Máscaras

Um kernel é uma pequena matriz utilizada para transformar uma imagem.

Exemplo de kernel de média:

$$
\frac{1}{9}
\begin{bmatrix}
1 & 1 & 1 \\
1 & 1 & 1 \\
1 & 1 & 1
\end{bmatrix}
$$

Esse filtro suaviza a imagem reduzindo variações locais.


In [ ]:

#| quarto-raw: true

kernel_media = np.ones((3,3), dtype=np.float32) / 9

print(kernel_media)



## Suavização por Média

Filtros de média reduzem ruídos ao substituir cada pixel pela média de sua vizinhança.


In [ ]:

#| label: fig-03-media
#| fig-cap: "Filtragem por média."
#| echo: true
#| output: true

img_media = cv2.filter2D(img_gray, -1, kernel_media)

fig, ax = plt.subplots(1,2, figsize=(12,5))

ax[0].imshow(img_gray, cmap='gray')
ax[0].set_title("Original")
ax[0].axis('off')

ax[1].imshow(img_media, cmap='gray')
ax[1].set_title("Filtro de média")
ax[1].axis('off')

plt.show()



## Filtro Gaussiano

O filtro Gaussiano produz suavização preservando melhor as estruturas da imagem.

Ele é amplamente utilizado antes da:
- detecção de bordas;
- segmentação;
- extração de características.


In [ ]:

#| label: fig-03-gauss
#| fig-cap: "Suavização Gaussiana."
#| echo: true
#| output: true

img_gauss = cv2.GaussianBlur(img_gray, (9,9), 0)

fig, ax = plt.subplots(1,2, figsize=(12,5))

ax[0].imshow(img_gray, cmap='gray')
ax[0].set_title("Original")
ax[0].axis('off')

ax[1].imshow(img_gauss, cmap='gray')
ax[1].set_title("Gaussiano")
ax[1].axis('off')

plt.show()



## Ruído em Imagens

Imagens digitais frequentemente apresentam ruídos causados por:
- sensores;
- transmissão;
- iluminação;
- compressão.

Um ruído muito comum é o ruído sal-e-pimenta.


In [ ]:

#| quarto-raw: true

def add_salt_pepper(img, prob=0.02):

    noisy = img.copy()

    rnd = np.random.rand(*img.shape)

    noisy[rnd < prob/2] = 0
    noisy[rnd > 1 - prob/2] = 255

    return noisy

img_noise = add_salt_pepper(img_gray)


In [ ]:

#| label: fig-03-ruido
#| fig-cap: "Imagem com ruído sal-e-pimenta."
#| echo: true
#| output: true

plt.imshow(img_noise, cmap='gray')
plt.axis('off')
plt.show()



## Filtro da Mediana

O filtro da mediana é muito eficiente para remover ruído impulsivo preservando bordas.


In [ ]:

#| label: fig-03-mediana
#| fig-cap: "Remoção de ruído utilizando filtro da mediana."
#| echo: true
#| output: true

img_mediana = cv2.medianBlur(img_noise, 5)

fig, ax = plt.subplots(1,2, figsize=(12,5))

ax[0].imshow(img_noise, cmap='gray')
ax[0].set_title("Com ruído")
ax[0].axis('off')

ax[1].imshow(img_mediana, cmap='gray')
ax[1].set_title("Filtro mediana")
ax[1].axis('off')

plt.show()



## Realce e Nitidez

Filtros passa-alta enfatizam transições abruptas de intensidade, aumentando detalhes e bordas.


In [ ]:

#| quarto-raw: true

kernel_sharp = np.array([
    [ 0, -1,  0],
    [-1,  5, -1],
    [ 0, -1,  0]
], dtype=np.float32)


In [ ]:

#| label: fig-03-sharp
#| fig-cap: "Realce de nitidez utilizando filtro passa-alta."
#| echo: true
#| output: true

img_sharp = cv2.filter2D(img_gray, -1, kernel_sharp)

fig, ax = plt.subplots(1,2, figsize=(12,5))

ax[0].imshow(img_gray, cmap='gray')
ax[0].set_title("Original")
ax[0].axis('off')

ax[1].imshow(img_sharp, cmap='gray')
ax[1].set_title("Nitidez")
ax[1].axis('off')

plt.show()



## Detecção de Bordas

Bordas representam mudanças abruptas de intensidade.

Elas são fundamentais em:
- segmentação;
- reconhecimento de objetos;
- visão computacional;
- análise estrutural.



### Gradiente e Operador de Sobel

O operador de Sobel estima derivadas horizontais e verticais da imagem.

Kernel horizontal:

$$
G_x=
\\begin{bmatrix}
-1 & 0 & 1 \\\\
-2 & 0 & 2 \\\\
-1 & 0 & 1
\\end{bmatrix}
$$

Kernel vertical:

$$
G_y=
\\begin{bmatrix}
-1 & -2 & -1 \\\\
0 & 0 & 0 \\\\
1 & 2 & 1
\\end{bmatrix}
$$


In [ ]:

#| label: fig-03-sobel
#| fig-cap: "Detecção de bordas utilizando Sobel."
#| echo: true
#| output: true

sobelx = cv2.Sobel(img_gray, cv2.CV_64F, 1, 0, ksize=3)
sobely = cv2.Sobel(img_gray, cv2.CV_64F, 0, 1, ksize=3)

magnitude = np.sqrt(sobelx**2 + sobely**2)

fig, ax = plt.subplots(1,3, figsize=(15,5))

ax[0].imshow(np.abs(sobelx), cmap='gray')
ax[0].set_title("Sobel X")
ax[0].axis('off')

ax[1].imshow(np.abs(sobely), cmap='gray')
ax[1].set_title("Sobel Y")
ax[1].axis('off')

ax[2].imshow(magnitude, cmap='gray')
ax[2].set_title("Magnitude")
ax[2].axis('off')

plt.show()



## Detector de Bordas de Canny

O detector de Canny é um dos métodos mais robustos para detecção de bordas.

Etapas:
1. suavização Gaussiana;
2. cálculo do gradiente;
3. supressão de não máximos;
4. limiarização por histerese.


In [ ]:

#| label: fig-03-canny
#| fig-cap: "Detecção de bordas utilizando Canny."
#| echo: true
#| output: true

edges = cv2.Canny(img_gray, 100, 200)

plt.imshow(edges, cmap='gray')
plt.axis('off')
plt.show()



## Comparação entre Filtros

Cada filtro possui características específicas:
- média: simples e rápido;
- Gaussiano: suavização natural;
- mediana: robusto para ruído impulsivo;
- passa-alta: realce de detalhes.



## Resumo

Neste capítulo foram apresentados:
- convolução bidimensional;
- kernels espaciais;
- filtragem linear;
- suavização;
- redução de ruído;
- realce de nitidez;
- detecção de bordas.

Esses conceitos formam a base matemática e computacional de grande parte do Processamento Digital de Imagens moderno.



## Uso do NotebookLM como Tutor Complementar

Sugestões de estudo:
- peça explicações sobre convolução;
- solicite exemplos de kernels;
- compare filtros lineares e não lineares;
- revise Sobel e Canny;
- explore aplicações práticas em visão computacional.



## Lista de Exercícios

1. Implementar convolução manualmente utilizando NumPy;
2. Comparar média e Gaussiano;
3. Testar diferentes tamanhos de kernel;
4. Adicionar ruídos artificiais;
5. Comparar filtros para remoção de ruído;
6. Implementar Sobel sem OpenCV;
7. Detectar bordas em imagens reais;
8. Comparar Sobel e Canny.



## Parte Prática com Exercícios de Programação

### Objetivo deste Caderno

Os exercícios a seguir consolidam os conceitos estudados neste capítulo através de implementações práticas utilizando Python.


### EP03_01 — Convolução Manual 2D

In [ ]:

# Escreva sua solução aqui



### EP03_02 — Filtro da Média

In [ ]:

# Escreva sua solução aqui



### EP03_03 — Filtro Gaussiano

In [ ]:

# Escreva sua solução aqui



### EP03_04 — Ruído Sal-e-Pimenta

In [ ]:

# Escreva sua solução aqui



### EP03_05 — Filtro da Mediana

In [ ]:

# Escreva sua solução aqui



### EP03_06 — Kernel de Nitidez

In [ ]:

# Escreva sua solução aqui



### EP03_07 — Detector de Bordas Sobel

In [ ]:

# Escreva sua solução aqui



### EP03_08 — Magnitude do Gradiente

In [ ]:

# Escreva sua solução aqui



### EP03_09 — Detector de Bordas Canny

In [ ]:

# Escreva sua solução aqui



### EP03_10 — Comparação entre Filtros

In [ ]:

# Escreva sua solução aqui

